# Text Classification using Deep Learning and Word Embeddings

References:
- https://www.geeksforgeeks.org/nlp/word-embeddings-in-nlp/
- https://www.geeksforgeeks.org/python/python-word-embedding-using-word2vec/
- https://www.geeksforgeeks.org/nlp/glove-word-embedding-in-nlp/
- https://www.geeksforgeeks.org/nlp/word-embeddings-using-fasttext/

## Data Loading & Exploration

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import random
import warnings

os.environ["PYTHONHASHSEED"] = "42"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import gensim.downloader as api
from gensim.models import Word2Vec, FastText

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import accuracy_score, f1_score

# Konfigurasi global agar seluruh proses reproducible
RANDOM_STATE = 42
EMBEDDING_DIM = 100
MAX_NUM_WORDS = 40000
MAX_LEN = 50
NUM_CLASSES = 3
BATCH_SIZE = 256
MAX_EPOCHS = 20
EMB_EPOCHS = 12               # embedding sendiri dilatih lebih lama (korpus train + test)

# Konfigurasi prediksi akhir (berdasarkan hasil audit)
N_SPLITS = 5
ENSEMBLE_SEEDS = [42, 202, 777]   # multi-seed: sumber diversity utama ensemble (audit poin 12)
DUP_OVERRIDE = True               # audit poin 1-3: teks uji identik dgn train -> pakai label train (faktor terbesar)

# Mengunci seed pada seluruh sumber keacakan
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

# cek ketersediaan GPU (di Kaggle aktifkan Settings -> Accelerator -> GPU)
print("GPU terdeteksi:", tf.config.list_physical_devices("GPU"))

In [5]:
import glob

# cari file dataset di lingkungan Kaggle (/kaggle/input) atau di direktori kerja; kalau tidak ketemu baru unduh dari Google Drive
def _cari_dataset(nama):
    kandidat = glob.glob("/kaggle/input/**/" + nama, recursive=True) + glob.glob(nama)
    return kandidat[0] if kandidat else None

_train_path = _cari_dataset("train.csv")
_test_path = _cari_dataset("test.csv")

if _train_path and _test_path:
    train_df = pd.read_csv(_train_path)
    test_df = pd.read_csv(_test_path)
else:
    train_df = pd.read_csv("https://drive.google.com/uc?id=1ln0z26Sod5nFuCrul6THQ12s9IiCUGoa")
    test_df = pd.read_csv("https://drive.google.com/uc?id=1IFM1Hn2zojj7EJdMIPHyON5o3S2rZjm9")

In [6]:
train_df.head()

,id,Text,Sentiment
0,1,modi hadnt become represented varanasi would r...,0
1,2,haha dont believe fake statistics taught feku ...,0
2,3,modi corruption scamhis family doesnt swiss ac...,0
3,4,bcos modi,1
4,5,rainy day rainy,1


In [7]:
test_df.head()

,id,Text
0,168802,disagree tejasvi surya isnâ replica modi see...
1,168803,stalin doesnt know even read correctly neither...
2,168804,result first pmwho abused modi chillar team
3,168805,get fingerprints checked porbably modis
4,168806,unlike cs introductions crash course great job...


## Text Preprocessing

In [8]:
# TODO: process the dataset using the techniques from the previous Lab Work or other techniques (explain in the report).

# perbaiki mojibake DULU (mis. "isnâ€™t"/"isn\x80\x99t" -> "isn't") sebelum tahap lain,
# supaya negasi tidak hancur menjadi token yatim "isn"/"didn" (audit poin 5/6)
try:
    from ftfy import fix_text as _fix_text
    _PUNYA_FTFY = True
except Exception:
    _PUNYA_FTFY = False

def _perbaiki_encoding(s):
    s = str(s)
    if _PUNYA_FTFY:
        return _fix_text(s)
    # fallback bila ftfy tak tersedia: petakan urutan mojibake apostrof/kutip yang umum
    for a, b in [("â€™", "'"), ("â€˜", "'"), ("â€œ", '"'), ("â€\x9d", '"'),
                 ("\x80\x99", "'"), ("\x80\x98", "'")]:
        s = s.replace(a, b)
    return s

# kontraksi tanpa apostrof (setelah apostrof dibuang cleaning); utamakan menjaga kata negasi
KONTRAKSI = {
    "dont": "do not", "didnt": "did not", "doesnt": "does not", "cant": "can not",
    "cannot": "can not", "wont": "will not", "wouldnt": "would not", "couldnt": "could not",
    "shouldnt": "should not", "isnt": "is not", "arent": "are not", "wasnt": "was not",
    "werent": "were not", "hasnt": "has not", "havent": "have not", "hadnt": "had not",
    "aint": "is not", "neednt": "need not", "mustnt": "must not", "mightnt": "might not",
    "wouldve": "would have", "couldve": "could have", "shouldve": "should have",
}
# emoji sentimen dipetakan ke token kata sebelum karakter non-alfabet dibuang
EMOJI = {
    "\U0001F602": " emojilucu ", "\U0001F923": " emojilucu ", "\U0001F60A": " emojisenang ",
    "\U0001F601": " emojisenang ", "\U0001F60D": " emojicinta ", "❤": " emojicinta ",
    "\U0001F44D": " emojibagus ", "\U0001F525": " emojibagus ", "\U0001F621": " emojimarah ",
    "\U0001F620": " emojimarah ", "\U0001F622": " emojisedih ", "\U0001F62D": " emojisedih ",
    "\U0001F44E": " emojijelek ",
}

# cleaning: perbaiki encoding -> huruf kecil -> jaga emoji -> buang non-alfanumerik -> elongasi -> kontraksi
def bersihkan_teks(teks):
    teks = _perbaiki_encoding(teks).lower()
    for emo, tok in EMOJI.items():                              # jaga sinyal emoji jadi token kata
        teks = teks.replace(emo, tok)
    teks = re.sub(r"[^a-z0-9\s]", " ", teks)                    # buang non-alfanumerik + sisa encoding
    teks = re.sub(r"(.)\1{2,}", r"\1\1", teks)                  # normalisasi elongasi: gooood -> good
    teks = " ".join(KONTRAKSI.get(kata, kata) for kata in teks.split())  # ekspansi kontraksi, jaga "not"
    teks = re.sub(r"\s+", " ", teks).strip()
    return teks

# buat nilai kosong pada kolom teks (baris uji tidak boleh dibuang agar jumlah prediksi tetap)
train_df["Text"] = train_df["Text"].fillna("").astype(str)
test_df["Text"] = test_df["Text"].fillna("").astype(str)
train_df["clean_text"] = train_df["Text"].apply(bersihkan_teks)
test_df["clean_text"] = test_df["Text"].apply(bersihkan_teks)

# tokenizer hanya di fit pada data latih untuk menghindari kebocoran informasi dari data uji
tokenizer = Tokenizer(num_words=MAX_NUM_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["clean_text"])
word_index = tokenizer.word_index
VOCAB_SIZE = min(MAX_NUM_WORDS, len(word_index) + 1)

# ubah teks menjadi urutan indeks lalu memadankan panjang (padding) di akhir kalimat
X_full = pad_sequences(tokenizer.texts_to_sequences(train_df["clean_text"]),
                       maxlen=MAX_LEN, padding="post", truncating="post")
X_test = pad_sequences(tokenizer.texts_to_sequences(test_df["clean_text"]),
                       maxlen=MAX_LEN, padding="post", truncating="post")
y_full = train_df["Sentiment"].values

# korpus token tanpa label untuk melatih Word2Vec & FastText: gabung train + test
# (label tidak dipakai, jadi bukan kebocoran) agar vektor juga mencakup kosakata data uji
korpus_teks = pd.concat([train_df["clean_text"], test_df["clean_text"]], ignore_index=True)
train_sentences = [teks.split() for teks in korpus_teks if teks.strip()]

X_tr, X_val, y_tr, y_val = train_test_split(
    X_full, y_full, test_size=0.1, random_state=RANDOM_STATE, stratify=y_full
)

print("ftfy tersedia:", _PUNYA_FTFY, "| Ukuran kosakata (VOCAB_SIZE):", VOCAB_SIZE)
print("Jumlah kalimat korpus embedding (train + test):", len(train_sentences))
print("Bentuk data latih:", X_tr.shape, "| validasi:", X_val.shape, "| uji:", X_test.shape)
print("Distribusi label pada data latih:", np.bincount(y_tr))

ftfy tersedia: False | Ukuran kosakata (VOCAB_SIZE): 40000
Jumlah kalimat korpus embedding (train + test): 240667
Bentuk data latih: (151920, 50) | validasi: (16881, 50) | uji: (72344, 50)
Distribusi label pada data latih: [34722 52272 64926]


## Feature Extraction

### Word Embedding

Word Embedding is an approach for representing words and documents. Word Embedding or Word Vector is a numeric vector input that represents a word in a lower-dimensional space.
Usage:
- To reduce dimensionality.
- To use a word to predict the words around it.
- Helps in enhancing model interpretability due to numerical representation.
- Inter-word semantics and similarity can be captured.

### Word2Vec

Word2Vec is a word embedding technique in NLP that represents words as vectors in a continuous space. Developed by Google, it captures semantic relationships by assigning similar vectors to words with similar meanings.

- Converts words into numerical vectors for machine learning models
- Captures semantic relationships between words
- Words with similar meanings have similar vector representations
- Developed by Google researchers
- Uses two main architectures: CBOW (Continuous Bag of Words) and Skip-Gram

In [9]:
# TODO: Use Word2Vec

# train Word2Vec (skip-gram) pada korpus latih; workers=1 + seed dipakai agar hasil reproducible
word2vec_model = Word2Vec(
    sentences=train_sentences,
    vector_size=EMBEDDING_DIM,
    window=5,
    min_count=2,
    sg=1,
    negative=10,
    workers=1,
    seed=RANDOM_STATE,
    epochs=EMB_EPOCHS,
)

# nyusun matriks embedding sesuai indeks kata pada tokenizer;
# dimensi diambil dari vektor sumber agar mendukung embedding pra-latih berdimensi lain (mis. GloVe 200)
def bangun_matriks_embedding(key_vectors, allow_oov=False):
    dim = key_vectors.vector_size
    matriks = np.zeros((VOCAB_SIZE, dim), dtype="float32")
    jumlah_terisi = 0
    for kata, indeks in word_index.items():
        if indeks >= VOCAB_SIZE:
            continue
        if kata in key_vectors.key_to_index:
            matriks[indeks] = key_vectors[kata]
            jumlah_terisi += 1
        elif allow_oov:
            try:
                matriks[indeks] = key_vectors[kata]
                jumlah_terisi += 1
            except KeyError:
                pass
    return matriks, jumlah_terisi

embedding_matrix_word2vec, terisi_w2v = bangun_matriks_embedding(word2vec_model.wv)
print("Word2Vec - dimensi matriks:", embedding_matrix_word2vec.shape,
      "cakupan kosakata: %.2f%%" % (100 * terisi_w2v / (VOCAB_SIZE - 1)))

Word2Vec - dimensi matriks: (40000, 100) cakupan kosakata: 100.00%


### GloVe

GloVe (Global Vectors for Word Representation) is an unsupervised learning algorithm that generates dense word embeddings by analyzing co-occurrence patterns in a large text corpus, capturing semantic relationships between words.

- Uses a word co-occurrence matrix to learn relationships between words
- Combines global statistical information (LSA) with local context-based learning (like Word2Vec)
- Optimizes embeddings so the dot product approximates Pointwise Mutual Information (PMI)
- Captures both semantic and syntactic relationships

In [10]:
# TODO: Load pre-trained GloVe

# Unduh GloVe pra-latih Twitter langsung dari rilis gensim-data dengan RESUME + retry.
# Masalah ContentTooShortError terjadi karena berkas besar (~758 MB) putus di tengah dan
# gensim MENGULANG dari nol tiap percobaan. Di sini kita pakai HTTP Range agar unduhan
# DILANJUTKAN dari byte terakhir, verifikasi ukuran, lalu muat via KeyedVectors (format word2vec).
import os, time, requests
from gensim.models import KeyedVectors

_GLOVE_DIR = getattr(api, "base_dir", os.path.expanduser(os.path.join("~", "gensim-data")))
_RILIS = getattr(api, "DOWNLOAD_BASE_URL",
                 "https://github.com/RaRe-Technologies/gensim-data/releases/download")

def _unduh_resumable(url, tujuan, percobaan=20, timeout=60):
    # ukuran total dari server (untuk tahu kapan unduhan sudah lengkap)
    total = None
    try:
        h = requests.head(url, allow_redirects=True, timeout=timeout)
        if h.headers.get("Content-Length"):
            total = int(h.headers["Content-Length"])
    except Exception:
        pass
    for i in range(1, percobaan + 1):
        ada = os.path.getsize(tujuan) if os.path.exists(tujuan) else 0
        if total is not None and ada >= total:
            return True
        headers = {"Range": "bytes=%d-" % ada} if ada else {}
        try:
            with requests.get(url, headers=headers, stream=True, timeout=timeout) as r:
                r.raise_for_status()
                # jika server abai pada Range (200, bukan 206), mulai ulang dari nol
                mode = "ab" if (ada and r.status_code == 206) else "wb"
                if mode == "wb":
                    ada = 0
                with open(tujuan, mode) as f:
                    for potongan in r.iter_content(1 << 20):
                        if potongan:
                            f.write(potongan)
                            ada += len(potongan)
            if total is None or ada >= total:
                return True
            print("  parsial %d/%d byte (percobaan %d/%d), melanjutkan..." % (ada, total, i, percobaan))
        except Exception as e:
            print("  unduhan terputus (percobaan %d/%d): %s" % (i, percobaan, e))
            time.sleep(3)
    return False

def _muat_glove(nama):
    folder = os.path.join(_GLOVE_DIR, nama)
    os.makedirs(folder, exist_ok=True)
    berkas = os.path.join(folder, nama + ".gz")
    url = "%s/%s/%s.gz" % (_RILIS, nama, nama)
    try:
        if _unduh_resumable(url, berkas):
            return KeyedVectors.load_word2vec_format(berkas)
    except Exception as e:
        print("Gagal memuat %s: %s" % (nama, e))
        try:
            os.remove(berkas)  # buang berkas rusak agar percobaan berikutnya bersih
        except OSError:
            pass
    return None

# 200-dim (lebih kaya); bila tetap gagal, cadangan turun ke 100-dim
glove_vectors = _muat_glove("glove-twitter-200") or _muat_glove("glove-twitter-100")
if glove_vectors is None:
    raise RuntimeError("Gagal mengunduh GloVe setelah beberapa percobaan; jalankan ulang sel ini.")

embedding_matrix_glove, terisi_glove = bangun_matriks_embedding(glove_vectors)
print("GloVe - dimensi matriks:", embedding_matrix_glove.shape,
      "cakupan kosakata: %.2f%%" % (100 * terisi_glove / (VOCAB_SIZE - 1)))

GloVe - dimensi matriks: (40000, 200) cakupan kosakata: 79.50%


### FastText

FastText is a word embedding technique developed by Facebook AI Research (FAIR) that represents words using character-level subwords (n-grams). This enables it to generate meaningful embeddings for rare and unseen words more effectively than traditional word embedding methods.

- Uses character-level subwords (n-grams) to represent words.
- Generates embeddings for rare and out-of-vocabulary (OOV) words.
- Supports both CBOW and Skip-Gram training methods.
Works well with morphologically rich languages.

In [11]:
# TODO: Use FastText

# train FastText (skip-gram) yang memakai sub-kata (n-gram) sehingga tetap memberi vektor untuk kata yg OOV
fasttext_model = FastText(
    sentences=train_sentences,
    vector_size=EMBEDDING_DIM,
    window=5,
    min_count=2,
    sg=1,
    negative=10,
    workers=1,
    seed=RANDOM_STATE,
    epochs=EMB_EPOCHS,
    min_n=3,
    max_n=6,
)

embedding_matrix_fasttext, terisi_ft = bangun_matriks_embedding(fasttext_model.wv, allow_oov=True)
print("FastText - dimensi matriks:", embedding_matrix_fasttext.shape,
      "cakupan kosakata: %.2f%%" % (100 * terisi_ft / (VOCAB_SIZE - 1)))

FastText - dimensi matriks: (40000, 100) cakupan kosakata: 100.00%


### SVD (Singular Value Decomposition)

SVD is a Dimensionality Reduction technique that can be used for NLP word embeddings.

Singular Value Decomposition (SVD) can be applied to reduce the dimensionality of the co-occurrence matrix and capture latent semantic relationships between words.

In [ ]:
# TODO: Create Word Embeddings using SVD

# susun daftar kata sesuai indeks tokenizer sebagai kosakata untuk matriks ko-okurensi
kosakata_svd = [""] * (VOCAB_SIZE - 1)
for kata, indeks in word_index.items():
    if indeks < VOCAB_SIZE:
        kosakata_svd[indeks - 1] = kata

# membangun matriks kata-kata (word-word) dari kemunculan bersama dalam satu dokumen
count_vectorizer = CountVectorizer(vocabulary=kosakata_svd, binary=True,
                                   token_pattern=r"(?u)\S+", lowercase=False)
dokumen_kata = count_vectorizer.fit_transform(train_df["clean_text"]).astype("float32")
ko_okurensi = (dokumen_kata.T @ dokumen_kata).tocsr()

# reduksi dimensi matriks dengan Truncated SVD menjadi embedding kata
svd = TruncatedSVD(n_components=EMBEDDING_DIM, random_state=RANDOM_STATE)
vektor_kata_svd = svd.fit_transform(ko_okurensi).astype("float32")
embedding_matrix_svd = np.zeros((VOCAB_SIZE, EMBEDDING_DIM), dtype="float32")
for kata, indeks in word_index.items():
    if indeks < VOCAB_SIZE:
        embedding_matrix_svd[indeks] = vektor_kata_svd[count_vectorizer.vocabulary_[kata]]

print("SVD - dimensi matriks:", embedding_matrix_svd.shape,
      "variansi terjelaskan: %.2f%%" % (100 * svd.explained_variance_ratio_.sum()))

SVD - dimensi matriks: (40000, 100) variansi terjelaskan: 95.26%


## Modelling: Deep Learning Algorithms

### CNN

In [13]:
# TODO: Build and train a CNN Model

embedding_matrices = {
    "Word2Vec": embedding_matrix_word2vec,
    "GloVe": embedding_matrix_glove,
    "FastText": embedding_matrix_fasttext,
    "SVD": embedding_matrix_svd,
    # gabungan GloVe (statistik global) + FastText (sub-kata, tahan OOV) -> 200 dimensi
    "GloVe+FastText": np.concatenate([embedding_matrix_glove, embedding_matrix_fasttext], axis=1),
}
hasil_evaluasi = []

# embedding diinisialisasi dari matriks pra-latih lalu ikut dilatih (fine tuning);
# dimensi diambil dari bentuk matriks agar embedding gabungan 200-dim juga didukung
def lapisan_embedding(matriks_embedding):
    return layers.Embedding(
        input_dim=matriks_embedding.shape[0],
        output_dim=matriks_embedding.shape[1],
        weights=[matriks_embedding],
        trainable=True,
    )

# CNN teks dengan beberapa ukuran kernel (3, 4, 5) yang digabungkan
def bangun_cnn(matriks_embedding):
    masukan = layers.Input(shape=(MAX_LEN,))
    x = lapisan_embedding(matriks_embedding)(masukan)
    x = layers.SpatialDropout1D(0.3)(x)
    cabang = [layers.GlobalMaxPooling1D()(layers.Conv1D(128, k, activation="relu")(x))
              for k in (3, 4, 5)]
    x = layers.Concatenate()(cabang)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, activation="relu")(x)
    keluaran = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    model = models.Model(masukan, keluaran)
    model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    return model

# pelatihan + evaluasi terpakai ulang untuk seluruh arsitektur agar perbandingan adil
def latih_dan_evaluasi(fungsi_bangun, matriks_embedding, nama_arsitektur, nama_embedding):
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    model = fungsi_bangun(matriks_embedding)
    early_stop = EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)
    model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=MAX_EPOCHS,
              batch_size=BATCH_SIZE, callbacks=[early_stop], verbose=0)
    prediksi_val = model.predict(X_val, verbose=0).argmax(axis=1)
    akurasi = accuracy_score(y_val, prediksi_val)
    f1_makro = f1_score(y_val, prediksi_val, average="macro")
    hasil_evaluasi.append({"arsitektur": nama_arsitektur, "embedding": nama_embedding,
                           "val_accuracy": akurasi, "val_macro_f1": f1_makro})
    print("%-5s + %-14s -> val_accuracy=%.4f | val_macro_f1=%.4f" %
          (nama_arsitektur, nama_embedding, akurasi, f1_makro))
    return model

# bandingkan seluruh teknik embedding memakai arsitektur CNN yang sama
model_cnn_per_embedding = {}
for nama_embedding, matriks in embedding_matrices.items():
    model_cnn_per_embedding[nama_embedding] = latih_dan_evaluasi(
        bangun_cnn, matriks, "CNN", nama_embedding)

# embedding terbaik berdasarkan akurasi validasi pada CNN
hasil_cnn = [h for h in hasil_evaluasi if h["arsitektur"] == "CNN"]
embedding_terbaik = max(hasil_cnn, key=lambda h: h["val_accuracy"])["embedding"]
embedding_matrix_terbaik = embedding_matrices[embedding_terbaik]
model_cnn = model_cnn_per_embedding[embedding_terbaik]
print("\nEmbedding terbaik untuk CNN:", embedding_terbaik)

I0000 00:00:1789107490.990582      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789107490.993436      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1789107498.676054     216 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


CNN   + Word2Vec       -> val_accuracy=0.8591 | val_macro_f1=0.8515
CNN   + GloVe          -> val_accuracy=0.8622 | val_macro_f1=0.8551
CNN   + FastText       -> val_accuracy=0.8635 | val_macro_f1=0.8562
CNN   + SVD            -> val_accuracy=0.4274 | val_macro_f1=0.1996
CNN   + GloVe+FastText -> val_accuracy=0.8661 | val_macro_f1=0.8592

Embedding terbaik untuk CNN: GloVe+FastText


### RNN

In [14]:
# TODO: Build and train an RNN model

# RNN dua arah (SimpleRNN) memakai embedding terbaik hasil perbandingan pada CNN
def bangun_rnn(matriks_embedding):
    masukan = layers.Input(shape=(MAX_LEN,))
    x = lapisan_embedding(matriks_embedding)(masukan)
    x = layers.SpatialDropout1D(0.3)(x)
    x = layers.Bidirectional(layers.SimpleRNN(64))(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, activation="relu")(x)
    keluaran = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    model = models.Model(masukan, keluaran)
    model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    return model

model_rnn = latih_dan_evaluasi(bangun_rnn, embedding_matrix_terbaik, "RNN", embedding_terbaik)

RNN   + GloVe+FastText -> val_accuracy=0.8346 | val_macro_f1=0.8251


### LSTM

In [18]:
# TODO: Build and train an LSTM model

# LSTM dua arah memakai embedding terbaik hasil perbandingan pada CNN
def bangun_lstm(matriks_embedding):
    masukan = layers.Input(shape=(MAX_LEN,))
    x = lapisan_embedding(matriks_embedding)(masukan)
    x = layers.SpatialDropout1D(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(64))(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, activation="relu")(x)
    keluaran = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    model = models.Model(masukan, keluaran)
    model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    return model

model_lstm = latih_dan_evaluasi(bangun_lstm, embedding_matrix_terbaik, "LSTM", embedding_terbaik)

# seluruh eksperimen (kombinasi embedding x arsitektur) diurutkan dari akurasi tertinggi
tabel_hasil = pd.DataFrame(hasil_evaluasi).sort_values("val_accuracy", ascending=False).reset_index(drop=True)
print(tabel_hasil.to_string(index=False))

LSTM  + GloVe+FastText -> val_accuracy=0.8593 | val_macro_f1=0.8516
arsitektur      embedding  val_accuracy  val_macro_f1
       CNN GloVe+FastText      0.866122      0.859220
       CNN       FastText      0.863515      0.856235
       CNN          GloVe      0.862153      0.855108
      LSTM GloVe+FastText      0.859309      0.851650
      LSTM GloVe+FastText      0.859191      0.851526
       CNN       Word2Vec      0.859132      0.851459
       RNN GloVe+FastText      0.834607      0.825062
       CNN            SVD      0.427404      0.199618


## Inference/Prediction and Kaggle Submission

In [ ]:
# TODO: Generate predictions on test.csv and save to submission format (you can use the example code below if you want)

# Strategi inferensi:
#  - embedding gabungan GloVe-200 + FastText-100 (300-dim), kombinasi terbaik pada validasi;
#  - ensemble CNN dgn bagging StratifiedKFold + multi-seed (RNN/LSTM/GRU tidak menambah akurasi);
#  - ReduceLROnPlateau + latih lebih lama; tuning pengali per-kelas pada OOF;
#  - DUPLICATE-AWARE OVERRIDE: teks uji identik dgn train memakai label train (faktor terbesar).
embedding_matrix_final = np.concatenate(
    [embedding_matrix_glove, embedding_matrix_fasttext], axis=1)
arsitektur_ensemble = [("CNN", bangun_cnn)]

# latih tiap fold x seed; rata-rata probabilitas pada X_target; kumpulkan OOF untuk tuning kelas
def latih_ensemble(X_data, y_data, X_target, seeds, arsitektur, n_splits=N_SPLITS,
                   kembalikan_oof=False):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    kumpulan_prob = []
    oof = np.zeros((len(X_data), NUM_CLASSES), dtype="float32")
    oof_hitung = np.zeros(len(X_data), dtype="float32")
    for fold, (idx_tr, idx_va) in enumerate(skf.split(X_data, y_data), start=1):
        for nama, fungsi in arsitektur:
            for seed in seeds:
                tf.keras.utils.set_random_seed(seed)
                model = fungsi(embedding_matrix_final)
                callbacks = [
                    EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
                    ReduceLROnPlateau(monitor="val_accuracy", factor=0.5, patience=2, min_lr=1e-5),
                ]
                model.fit(X_data[idx_tr], y_data[idx_tr],
                          validation_data=(X_data[idx_va], y_data[idx_va]),
                          epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
                          callbacks=callbacks, verbose=0)
                kumpulan_prob.append(model.predict(X_target, verbose=0))
                if kembalikan_oof:
                    oof[idx_va] += model.predict(X_data[idx_va], verbose=0)
                    oof_hitung[idx_va] += 1
        print("  fold %d/%d selesai" % (fold, n_splits))
    prob_target = np.mean(kumpulan_prob, axis=0)
    if kembalikan_oof:
        return prob_target, oof / np.maximum(oof_hitung[:, None], 1.0)
    return prob_target

# pengali per-kelas (coordinate ascent) yang memaksimalkan akurasi pada OOF
def cari_pengali(oof, y, putaran=3):
    pengali = np.ones(NUM_CLASSES, dtype="float32")
    grid = np.linspace(0.6, 1.4, 17)
    akurasi = accuracy_score(y, oof.argmax(axis=1))
    for _ in range(putaran):
        for c in range(NUM_CLASSES):
            for g in grid:
                kand = pengali.copy(); kand[c] = g
                a = accuracy_score(y, (oof * kand).argmax(axis=1))
                if a > akurasi:
                    akurasi, pengali = a, kand
    return pengali, akurasi

# ensemble final: CNN, bagging K-fold penuh + multi-seed, langsung pada seluruh data latih
print("Ensemble final (CNN, K-fold, multi-seed)...")
prob_final, oof_final = latih_ensemble(X_full, y_full, X_test, ENSEMBLE_SEEDS,
                                       arsitektur_ensemble, kembalikan_oof=True)

# tuning pengali per-kelas pada OOF untuk mengimbangi ketimpangan kelas
pengali, akurasi_oof = cari_pengali(oof_final, y_full)
print("Akurasi OOF: default %.4f -> setelah tuning kelas %.4f | pengali=%s" %
      (accuracy_score(y_full, oof_final.argmax(axis=1)), akurasi_oof, np.round(pengali, 3)))
predicted_classes = (prob_final * pengali).argmax(axis=1)

# teks uji yang identik dgn train memakai label train (exact dulu, lalu normalized; seri dilewati)
if DUP_OVERRIDE:
    from collections import Counter as _Counter, defaultdict as _dd
    def _norm(t):
        t = re.sub(r"[^a-z0-9\s]", " ", str(t).lower())
        return re.sub(r"\s+", " ", t).strip()
    peta_exact, peta_norm = _dd(list), _dd(list)
    for t, y in zip(train_df["Text"], y_full):
        peta_exact[str(t)].append(int(y))
        k = _norm(t)
        if k:
            peta_norm[k].append(int(y))
    def _vote(lst):
        c = _Counter(lst).most_common()
        return None if (len(c) > 1 and c[0][1] == c[1][1]) else c[0][0]
    pred_final = predicted_classes.copy(); n_exact = n_norm = 0
    for i, t in enumerate(test_df["Text"]):
        ts = str(t)
        if ts in peta_exact:
            v = _vote(peta_exact[ts])
            if v is not None:
                if pred_final[i] != v: n_exact += 1
                pred_final[i] = v; continue
        k = _norm(t)
        if k in peta_norm:
            v = _vote(peta_norm[k])
            if v is not None:
                if pred_final[i] != v: n_norm += 1
                pred_final[i] = v
    print("Override duplikat -> exact ubah %d, normalized tambah %d, total %d (%.2f%% test)" %
          (n_exact, n_norm, n_exact + n_norm, 100 * (n_exact + n_norm) / len(test_df)))
    predicted_classes = pred_final

# Format to match sample_submission.csv
submission_df = pd.DataFrame({
    "id": test_df["id"],
    "Sentiment": predicted_classes.astype(int),
})
submission_df.to_csv("submission.csv", index=False)
print("submission.csv tersimpan dengan", len(submission_df), "baris.")
print("Distribusi prediksi:", submission_df["Sentiment"].value_counts().sort_index().to_dict())